# MLflow 5 minute Tracking Quickstart

This notebook demonstrates using a local MLflow Tracking Server to log, register, and then load a model as a generic Python Function (pyfunc) to perform inference on a Pandas DataFrame.

Throughout this notebook, we'll be using the MLflow fluent API to perform all interactions with the MLflow Tracking Server.

This is a slight modification of the original MLflow tutorial.

### Set the MLflow Tracking URI 

Depending on where you are running this notebook, your configuration may vary for how you initialize the interface with the MLflow Tracking Server. 

For this example, we're using a locally running tracking server, but other options are available.  
(The easiest is to use the free managed service within the [Databricks Free Trial](https://mlflow.org/docs/latest/getting-started/databricks-trial.html)). 

Please see [the guide to running notebooks here](https://www.mlflow.org/docs/latest/getting-started/running-notebooks/) for more information on setting the tracking server uri and configuring access to either managed or self-managed MLflow tracking servers.

## MISSING FROM THE OFFICIAL TUTORIAL

First, we need to start an MLflow server from the command line:

```bash
mlflow ui --host 127.0.0.1 --port 8080  # or 8081 if 8080 is already in use
```

## MLflow Default Database and Registry store URI

When you launch `mlflow ui` (or `mlflow server`) without flags, MLflow has to pick defaults for **three** independent stores.   
Understanding what each one holds — and which defaults changed in MLflow 3 — keeps the rest of this tutorial from feeling like magic.

### The three stores

| Store | Holds | Default URI (MLflow 3+) |
|---|---|---|
| **Backend (tracking) store** | Experiments, runs, parameters, metrics, tags — all *metadata* | `sqlite:///mlflow.db` |
| **Artifact store** | Model files, plots, datasets, anything binary you log | `mlflow-artifacts:` (proxied) → `./mlartifacts/` on disk |
| **Model Registry store** | Registered models, versions, aliases, stages | Same URI as the backend store |

You only typically interact with these defaults by *not* setting them.   
Each can be overridden with a CLI flag (`--backend-store-uri`, `--default-artifact-root`, `--registry-store-uri`) or with an environment variable.

### MLflow Database (Backend Store)

**Default since MLflow 3+: `sqlite:///mlflow.db`**

The backend store is where MLflow records the *metadata* about your experiments and runs:
- the run id, 
- -start/end time, 
- parameters you logged,
- metrics,
- tags,
- status,
- the user, and so on.

It does **not** hold model files — those go to the artifact store.

What changed in MLflow 3: prior versions defaulted to `file:./mlruns`, a flat directory of YAML/text files (one folder per experiment, one per run). It worked, but it had two problems:

1. **No concurrent writes.** Two processes writing runs at the same time could corrupt the directory.
2. **No Model Registry support.** The registry needs SQL-style transactions (atomic version creation, foreign keys between models and runs).      
The file backend can't provide that, so `mlflow.register_model(...)` and `registered_model_name=...` simply failed with the file backend.

From MLflow 3 onward, `mlflow ui` creates a SQLite database file called `mlflow.db` in the working directory where you started the server. 

SQLite gives you transactions and lets the registry "just work" out of the box — which is exactly why the `registered_model_name="tracking-quickstart"` argument later in this notebook succeeds without any extra configuration.

**Where the file lives:** in the cwd of the `mlflow ui` process. If you start the server from the repo root, you'll see `mlflow.db` appear there.   

Add it to `.gitignore` — it's per-developer state, not source code.

**Inspecting it (optional):**
```bash
sqlite3 mlflow.db ".tables"
sqlite3 mlflow.db "SELECT experiment_id, name FROM experiments;"
```

**Overriding it:** pass `--backend-store-uri` to use Postgres / MySQL / a different SQLite path:
```bash
mlflow ui --backend-store-uri postgresql://user:pw@host:5432/mlflow
mlflow ui --backend-store-uri sqlite:///path/to/custom.db
```

### MLflow Registry Store

**Default: same URI as the backend store** (so also `sqlite:///mlflow.db` by default in MLflow 3+).

The Model Registry is a separate logical service that tracks *registered models* — named, versioned references to logged models, plus their aliases (e.g. `champion`, `challenger`) and lifecycle stage.    
A "run" logs a model artifact; "registering" it gives that artifact a stable name and version number you can promote, query, and load by name from production code.

The registry needs a database backend for the same transactional reasons as above.   
MLflow's design choice is: rather than configure two URIs, default the registry to **share** the backend's URI.   
Both end up as tables inside the same SQLite file (`registered_models`, `model_versions`, etc., alongside `experiments`, `runs`, `params`, `metrics`).

You can split them if you ever need to — e.g. a shared team-wide registry but a per-user tracking backend — by passing `--registry-store-uri` explicitly:
```bash
mlflow server \
  --backend-store-uri sqlite:///local-tracking.db \
  --registry-store-uri postgresql://team@registry-host/mlflow_registry
```
For local learning, leaving the defaults alone is the right call.

### Reading the server's startup log

The lines `mlflow ui` prints on startup map directly onto the table above:

```text
Backend store URI not provided. Using sqlite:///mlflow.db
Registry store URI not provided. Using backend store URI.
[MLflow] Security middleware enabled with default settings (localhost-only).
To allow connections from other hosts, use --host 0.0.0.0 and configure --allowed-hosts and --cors-allowed-origins.
```

- **Line 1** — backend store fell through to the MLflow 3+ default, SQLite.
- **Line 2** — registry store was not separately configured, so it shares the backend URI.
- **Lines 3–4** — unrelated to storage: MLflow 3+ binds the UI to `127.0.0.1` only and rejects requests with non-localhost `Host:` headers as a SSRF/CSRF hardening default. If you `--host 0.0.0.0` to expose the UI on a LAN, you also need to opt into trusted hostnames and origins.

### What you'll see on disk after this notebook runs

Start the mlflow server from the directroy named `src`, in the repo, and `mlflow.db` will be created:

```bash
❯ tree
.
├── CLAUDE.md
├── docs
├── pyproject.toml
├── README.md
├── src
│   ├── mlflow.db
│   └── official_tutorials
│       └── tracking_quickstart.ipynb
└── uv.lock
```

After running the notebook the project structure will be:

```bash
.............................
```

If you don't want any of these tracked in git, add them to `.gitignore`:

```gitignore
mlflow.db
mlartifacts/
mlruns/
```

### Sources

- [Use SQLite as default unless existing mlruns data is detected — mlflow/mlflow PR #18497](https://github.com/mlflow/mlflow/pull/18497)
- [Backend Stores — MLflow docs](https://mlflow.org/docs/latest/ml/tracking/backend-stores/)


In [1]:
import pandas as pd
from sklearn import datasets
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split

import mlflow
from mlflow.models import infer_signature

In [2]:
# NOTE: review the links mentioned above for guidance on connecting to a managed tracking server, such as the Databricks Managed MLflow
HOST = "127.0.0.1"
PORT = 8081
mlflow.set_tracking_uri(uri=f"http://{HOST}:{PORT}")

## Load training data and train a simple model

For our quickstart, we're going to be using the familiar iris dataset that is included in scikit-learn. Following the split of the data, we're going to train a simple logistic regression classifier on the training data and calculate some error metrics on our holdout test data. 

Note that the only MLflow-related activities in this portion are around the fact that we're using a `param` dictionary to supply our model's hyperparameters.   
This is to make logging these settings easier when we're ready to log our model and its associated metadata.

Nothing is logged to MLflow in this loading of data and in this training.

In [3]:
# Load the Iris dataset
X, y = datasets.load_iris(return_X_y=True)

# Split the data into training and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [4]:
# Define the model hyperparameters
# NOTE (differs from upstream tutorial): the original `params` dict includes
# `"multi_class": "auto"`. That keyword was deprecated in scikit-learn 1.5 and
# removed in 1.7, so it raises `TypeError` on scikit-learn >= 1.7. Modern
# LogisticRegression automatically uses multinomial for multiclass — drop the key.
params = {"solver": "lbfgs", "max_iter": 1000, "random_state": 8888}

# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

# Predict on the test set
y_pred = lr.predict(X_test)

# Calculate accuracy as a target loss metric
accuracy = accuracy_score(y_test, y_pred)

## Define an MLflow Experiment

In order to group any distinct runs of a particular project or idea together, we can define an Experiment that will group each iteration (runs) together.  

Define a unique name relevant to what we're working on, to help with organization and reduce the amount of work (searching) to find our runs later on. 

In [ ]:
experiment_name = "MLflow Quickstart"

mlflow.set_experiment(experiment_name)

In [6]:
mlflow.get_experiment_by_name(experiment_name)

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1778498051076, experiment_id='1', last_update_time=1778498051076, lifecycle_stage='active', name='MLflow Quickstart', tags={}, trace_location=None, workspace='default'>

## Log the model, hyperparameters, and loss metrics to MLflow.

To record:
- our model and 
- the hyperparameters that were used when fitting the model, 
- as well as the metrics associated with validating the fit model upon holdout data,
  
we initiate a run context, as shown below.  

Within the scope of that context, any fluent API that we call (such as `mlflow.log_params()` or `mlflow.sklearn.log_model()`) will be associated and logged together to the same run. 

**By default, this creates a folder `mlruns` relative to where the mlflow server was started.**  
  
I started it in the project root so the folder is on the same level as `src`.

In [7]:
# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("accuracy", accuracy)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model for iris data")

    # Infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    # Log the model
    # NOTE (differs from upstream tutorial): serialization_format="skops" avoids
    # pickle/cloudpickle, which execute arbitrary code on load. See the final
    # markdown cell of this notebook for the full explanation.
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        name="iris_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="tracking-quickstart",
        serialization_format="skops",
    )

2026/05/11 15:00:15 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'tracking-quickstart' already exists. Creating a new version of this model...
2026/05/11 15:00:16 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: tracking-quickstart, version 2
Created version '2' of model 'tracking-quickstart'.


🏃 View run blushing-rook-969 at: http://127.0.0.1:8081/#/experiments/1/runs/a5c0b1ff4bbd4754b918adc44e9a2152
🧪 View experiment at: http://127.0.0.1:8081/#/experiments/1


After the first run, the `mlartifacts' directory is created with the following structure:

```bash
❯ tree
.
├── CLAUDE.md
├── docs
├── pyproject.toml
├── README.md
├── src
│   ├── mlartifacts
│   │   └── 1
│   │       └── models
│   │           └── m-9712f7b345d5415c8ae921f4f2ba5c64
│   │               └── artifacts
│   │                   ├── conda.yaml
│   │                   ├── input_example.json
│   │                   ├── MLmodel
│   │                   ├── model.pkl
│   │                   ├── python_env.yaml
│   │                   ├── requirements.txt
│   │                   └── serving_input_example.json
│   ├── mlflow.db
│   └── official_tutorials
│       └── tracking_quickstart.ipynb
└── uv.lock
```

**While it can be valid to wrap the entire code (training and inference run) within the `start_run()` block, this is not recommended.**   

If there as in issue with the training of the model or any other portion of code that is unrelated to MLflow-related actions, an empty or partially-logged run will be created, which will necessitate manual cleanup of the invalid run.    

**It is best to keep the training execution outside of the run context block to ensure that the loggable content (parameters, metrics, artifacts, and the model) are fully materialized prior to logging.**

## Directory structure after second experiment run with the same name

```bash
❯ tree -c
.
├── docs
├── CLAUDE.md
├── README.md
├── pyproject.toml
├── uv.lock
└── src
    ├── mlartifacts
    │   └── 1
    │       └── models
    │           ├── m-9712f7b345d5415c8ae921f4f2ba5c64
    │           │   └── artifacts
    │           │       ├── conda.yaml
    │           │       ├── input_example.json
    │           │       ├── MLmodel
    │           │       ├── model.pkl
    │           │       ├── python_env.yaml
    │           │       ├── requirements.txt
    │           │       └── serving_input_example.json
    │           └── m-260866c0927a4086abb797184f736ea3
    │               └── artifacts
    │                   ├── conda.yaml
    │                   ├── input_example.json
    │                   ├── MLmodel
    │                   ├── model.skops
    │                   ├── python_env.yaml
    │                   ├── requirements.txt
    │                   └── serving_input_example.json
    ├── official_tutorials
    │   └── tracking_quickstart.ipynb
    └── mlflow.db

11 directories, 20 files
```

## Load our saved model as a Python Function

We can load our model back as a native scikit-learn format with `mlflow.sklearn.load_model()`.   

But,  we are loading the model as a generic Python Function, which is how this model would be loaded for online model serving.    

We can still use the `pyfunc` representation for batch use cases, though, as is shown below.

In [8]:
loaded_model = mlflow.pyfunc.load_model(model_info.model_uri)

## Use our model to predict the iris class type on a Pandas DataFrame

In [9]:
predictions = loaded_model.predict(X_test)

iris_feature_names = datasets.load_iris().feature_names

# Convert X_test validation feature data to a Pandas DataFrame
result = pd.DataFrame(X_test, columns=iris_feature_names)

# Add the actual classes to the DataFrame
result["actual_class"] = y_test

# Add the model predictions to the DataFrame
result["predicted_class"] = predictions

result[:4]

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),actual_class,predicted_class
0,6.1,2.8,4.7,1.2,1,1
1,5.7,3.8,1.7,0.3,0,0
2,7.7,2.6,6.9,2.3,2,2
3,6.0,2.9,4.5,1.5,1,1


## MISSING FROM THE OFFICIAL TUTORIAL: serializing with `skops` instead of pickle

When you run `mlflow.sklearn.log_model(...)` without specifying `serialization_format`, MLflow emits this warning:

```
WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle
format requires exercising caution because these formats rely on Python's object
serialization mechanism, which can execute arbitrary code during deserialization.
The recommended safe alternative is the 'skops' format.
```

This section explains **what the warning means**, **why it matters**, and **the one-line fix** applied in the cell above.

### Why pickle/cloudpickle are unsafe

`pickle` (and its superset `cloudpickle`, which adds support for lambdas, locally-defined classes, and other "tricky" Python objects) does not serialize *data* — it serializes a **byte-stream program** that, when read back, reconstructs the object by executing instructions on a virtual machine inside the unpickler.

Those instructions include opcodes like `GLOBAL` (look up any module/attribute by name), `REDUCE` (call any callable with any arguments), and `BUILD` (set arbitrary state on the result).   

In practice this means: **anyone who can hand you a `.pkl` file can run arbitrary Python on your machine the moment you call `mlflow.pyfunc.load_model(...)`** — no exploit, no parser bug; that *is* the documented behavior of `pickle.load`.

The Python docs are explicit about this:

> **Warning:** The `pickle` module **is not secure**. Only unpickle data you trust.

For a personal notebook on your own laptop this is fine — you trained the model, you saved it, you loaded it.   
But the moment a model moves between people or systems (a teammate pulls it from the MLflow registry, a CI job loads it from S3, you publish it to a public artifact store), pickle-based loading becomes a real attack surface.

### What `skops` does differently

`skops` is a serialization library from the scikit-learn org, built specifically to replace pickle for ML models. Its persistence format is fundamentally different:

| Property | `pickle` / `cloudpickle` | `skops` |
|---|---|---|
| On-disk shape | Byte-stream program | Zip archive: numpy arrays + JSON schema |
| Loading mechanism | Executes serialized opcodes | Reconstructs known types from a schema |
| Unknown types | Silently instantiated (RCE) | Refused unless explicitly trusted |
| File extension | `.pkl` | `.skops` |
| Diff-friendly | No (binary opcodes) | Mostly yes (JSON-ish schema) |

The key idea: skops loads from a **declarative description** of the object tree, not an imperative program.    

When the loader sees `sklearn.linear_model.LogisticRegression`, it checks that type against an allow-list of known-safe types and instantiates it. Anything **not** on the allow-list raises `UntrustedTypesFoundException` instead of being executed.

### The fix applied above

Two changes were needed:

**1. Add `skops` to the project dependencies (uv):**

```bash
uv add skops
```

This added `skops>=0.14.0` to `[project].dependencies` in `pyproject.toml` and recorded the resolved version in `uv.lock`.

**2. Pass `serialization_format="skops"` to `log_model`** (see the cell above). MLflow's `mlflow.sklearn.log_model` accepts three values for this argument: `"pickle"`, `"cloudpickle"` (the default), and `"skops"`.

That's the whole fix. For built-in scikit-learn estimators like `LogisticRegression`, the skops default allow-list already includes every required type, so `mlflow.pyfunc.load_model(...)` further down the notebook still works with no other changes.

### Caveat: custom classes

If your model contains any **non-scikit-learn class** — a custom `BaseEstimator` subclass, a custom `Transformer`, a pipeline step you defined yourself — `skops` will refuse to load it by default, because that class isn't in its allow-list. The escape hatch is the `skops_trusted_types` argument:

```python
mlflow.sklearn.log_model(
    sk_model=pipe,
    name="pipe_model",
    serialization_format="skops",
    skops_trusted_types=["__main__.MyCustomTransformer"],
)
```

This is *opt-in trust*, not "turn off all checks": you're stating, per fully-qualified type name, that you accept responsibility for that class being safe to instantiate. The iris model in this notebook is pure stock sklearn, so no `skops_trusted_types` list is needed.

### When you might *not* use `skops`

A few situations still call for cloudpickle (and accepting the warning consciously):

- **Models that pickle closures or lambdas** — skops's format is structural; it cannot serialize a function body. If your estimator captures a lambda, you'll get a `NotImplementedError` from skops and need cloudpickle.
- **Third-party estimators with deeply custom internals** — some libraries (e.g. certain gradient-boosting wrappers) store state that skops can't introspect.
- **Quick local experimentation where the model file never leaves your machine** — the security risk is zero, and pickle is one less dependency.

For everything else — and especially for any model that will live in the MLflow registry where other people or services will load it — `skops` is the correct default in MLflow 3+.

### References

- [`mlflow.sklearn` API — `serialization_format` parameter](https://mlflow.org/docs/latest/python_api/mlflow.sklearn.html)
- [Pickle-Free Model format — MLflow docs](https://mlflow.org/docs/latest/ml/tracking/pickle-free-models/)
- [`skops` project — secure scikit-learn serialization](https://skops.readthedocs.io/en/stable/persistence.html)


## MISSING FROM THE OFFICIAL TUTORIAL: naming the artifact directory with `artifact_location`

Earlier in the tutorial, after running the training cell twice under the same experiment, you saw this on-disk layout:

```text
mlartifacts/
└── 1/                              ← experiment_id, NOT experiment_name
    └── models/
        ├── m-9712f7b345d5...64/    ← LoggedModel UUID, first run (pickle)
        │   └── artifacts/
        │       └── model.pkl
        └── m-260866c0927a...a3/    ← LoggedModel UUID, second run (skops)
            └── artifacts/
                └── model.skops
```

This addendum explains:

1. **why** the parent directory is the integer `1` and not the experiment's name,
2. **which MLflow parameter** controls it (`artifact_location` on `create_experiment`), and
3. **how** to opt into a human-readable path on a *new* experiment without disturbing anything the main tutorial already wrote.

The main tutorial above is left intact on purpose — the integer-id layout is what MLflow gives you out of the box, and seeing it once is part of the lesson.

### Why the integer `1`?

When you called `mlflow.set_experiment("MLflow Quickstart")` earlier and that experiment didn't yet exist, MLflow created it and assigned the next available **integer id**. The built-in `Default` experiment is always `0`; yours was the second to exist on this tracking server, so it got `1`. The output of the `set_experiment` cell makes this explicit:

```text
<Experiment: artifact_location='mlflow-artifacts:/1', experiment_id='1',
             name='MLflow Quickstart', ...>
```

The `artifact_location` is the **per-experiment root** under which all runs and their models live.

MLflow uses `experiment_id` (not the name) in this path on purpose:

- Experiments can be renamed; the id is stable.
- Putting the name into the on-disk path would mean a rename has to move thousands of files.
- Using the id keeps rename a one-row `UPDATE` in `mlflow.db`.

### Why both runs shared `1/`

`mlflow.set_experiment("MLflow Quickstart")` was called once. Every subsequent `mlflow.start_run()` (re-executing the log-model cell) attached a new run to the *same* experiment, so:

- Same experiment → same `experiment_id` → same artifact root → same `1/` folder.

### Why each `m-<hash>` is a separate directory

In MLflow 3, every call to `mlflow.<flavor>.log_model(...)` creates a **`LoggedModel` entity** with its own UUID, separate from the run that produced it. That UUID is the `m-<hex>` in the path. Two runs that each log a model produce two `LoggedModels` with two different UUIDs and therefore two sibling directories under `models/`. In our case the only file that differs between them is `model.pkl` (first run, before the skops fix) vs `model.skops` (second run, after) — everything else (`MLmodel`, `conda.yaml`, `requirements.txt`, …) is regenerated per `LoggedModel`.


### The MLflow parameter responsible: `artifact_location`

The parameter that controls this on-disk path is **`artifact_location`** on `mlflow.create_experiment(name, artifact_location=..., tags=...)`.

- **If you don't pass it**, the server uses its **default-artifact-root** (`mlflow-artifacts:/` by default, which resolves to `./mlartifacts/` on disk where you launched `mlflow ui`) and appends the new experiment's id: `mlflow-artifacts:/<experiment_id>`. That's why we got `mlartifacts/1/`.
- **If you do pass it**, your value is used verbatim — no `experiment_id` is appended — and that's what ends up as the on-disk parent directory.

Two important constraints when applying this fix:

1. **`mlflow.set_experiment(name)` does NOT accept `artifact_location`.** It only creates the experiment with defaults if it's missing. To control the location you must call `mlflow.create_experiment(...)` explicitly *before* `set_experiment(...)`.
2. **`artifact_location` is set once, at creation, and is effectively immutable through the public API.** Renaming an experiment is fine; *relocating* one is not. So the existing `"MLflow Quickstart"` experiment from the main tutorial above, which already has `artifact_location='mlflow-artifacts:/1'`, will keep that path forever.

The cleanest demonstration is therefore to **create a brand-new experiment** with a friendly `artifact_location` and run training under it. The code cells below do exactly that.


In [10]:
# Create a new experiment with an explicit, human-readable artifact_location.
# This leaves the original "MLflow Quickstart" experiment (and its mlartifacts/1/
# directory) untouched — we are demonstrating the parameter, not migrating data.
friendly_experiment_name = "MLflow Quickstart (friendly path)"

# `mlflow.create_experiment` raises if the experiment already exists, so we look
# it up first and only create it on the first run of this cell.
if mlflow.get_experiment_by_name(friendly_experiment_name) is None:
    mlflow.create_experiment(
        name=friendly_experiment_name,
        artifact_location="mlflow-artifacts:/mlflow-quickstart-friendly",
    )

# `set_experiment` makes it the active experiment for subsequent `start_run` calls.
mlflow.set_experiment(friendly_experiment_name)

# Inspect the experiment object — note artifact_location is the friendly path,
# not `mlflow-artifacts:/<experiment_id>`.
mlflow.get_experiment_by_name(friendly_experiment_name)


<Experiment: artifact_location='mlflow-artifacts:/mlflow-quickstart-friendly', creation_time=1778504234848, experiment_id='2', last_update_time=1778504234848, lifecycle_stage='active', name='MLflow Quickstart (friendly path)', tags={}, trace_location=None, workspace='default'>

### Demonstration: log a model under the new experiment

Re-run the same iris training and `log_model` under the friendly experiment. The training code below is identical to the main tutorial's run cell except that there's no `registered_model_name=` — the goal here is only to show the on-disk layout change, not to add another version to the `tracking-quickstart` registered model.


In [11]:
with mlflow.start_run(run_name="iris-friendly-path-demo"):
    mlflow.log_params(params)
    mlflow.log_metric("accuracy", accuracy)
    mlflow.set_tag("Training Info", "Friendly artifact_location demo")

    signature = infer_signature(X_train, lr.predict(X_train))

    friendly_model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        name="iris_model",
        signature=signature,
        input_example=X_train,
        serialization_format="skops",
    )

print("Model URI:", friendly_model_info.model_uri)


2026/05/11 15:57:21 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run iris-friendly-path-demo at: http://127.0.0.1:8081/#/experiments/2/runs/cf7c213657934240aa7f2e18d31aa905
🧪 View experiment at: http://127.0.0.1:8081/#/experiments/2
Model URI: models:/m-1647841a96d34450a9b4b7c7783c26ce


### Expected directory tree

After running the two cells above, `mlartifacts/` should now contain a sibling directory next to the original `1/`:

```text
❯ tree
.
├── CLAUDE.md
├── docs
├── pyproject.toml
├── README.md
├── src
│   ├── mlartifacts
│   │   ├── 1
│   │   │   └── models
│   │   │       ├── m-260866c0927a4086abb797184f736ea3
│   │   │       │   └── artifacts
│   │   │       │       ├── conda.yaml
│   │   │       │       ├── input_example.json
│   │   │       │       ├── MLmodel
│   │   │       │       ├── model.skops
│   │   │       │       ├── python_env.yaml
│   │   │       │       ├── requirements.txt
│   │   │       │       └── serving_input_example.json
│   │   │       └── m-9712f7b345d5415c8ae921f4f2ba5c64
│   │   │           └── artifacts
│   │   │               ├── conda.yaml
│   │   │               ├── input_example.json
│   │   │               ├── MLmodel
│   │   │               ├── model.pkl
│   │   │               ├── python_env.yaml
│   │   │               ├── requirements.txt
│   │   │               └── serving_input_example.json
│   │   └── mlflow-quickstart-friendly
│   │       └── models
│   │           └── m-1647841a96d34450a9b4b7c7783c26ce
│   │               └── artifacts
│   │                   ├── conda.yaml
│   │                   ├── input_example.json
│   │                   ├── MLmodel
│   │                   ├── model.skops
│   │                   ├── python_env.yaml
│   │                   ├── requirements.txt
│   │                   └── serving_input_example.json
│   ├── mlflow.db
│   └── official_tutorials
│       └── tracking_quickstart.ipynb
└── uv.lock

15 directories, 27 files
```

The MLflow UI at `http://127.0.0.1:8081/` now lists three experiments:

| `experiment_id` | name | `artifact_location` |
|---|---|---|
| `0` | `Default` | `mlflow-artifacts:/0` |
| `1` | `MLflow Quickstart` | `mlflow-artifacts:/1` |
| `2` | `MLflow Quickstart (friendly path)` | `mlflow-artifacts:/mlflow-quickstart-friendly` |

Note that the **`experiment_id` is still `2`** for the new experiment — `artifact_location` only changes the *filesystem* path, not the integer id. Anywhere MLflow's APIs return an experiment id (run URLs, the registry's underlying tables), you'll still see numbers.


### Alternatives if you really want to keep the original name

Two heavier options, both with caveats:

1. **Soft-delete the existing experiment, then create a new one with the same name and a friendly `artifact_location`:**

   ```python
   exp = mlflow.get_experiment_by_name("MLflow Quickstart")
   if exp is not None:
       mlflow.delete_experiment(exp.experiment_id)  # soft delete
   mlflow.create_experiment(
       name="MLflow Quickstart",
       artifact_location="mlflow-artifacts:/mlflow-quickstart",
   )
   ```

   `delete_experiment` is a soft-delete — runs and metrics stay in the SQLite backend (`lifecycle_stage = 'deleted'`) and the artifact files stay on disk. Worse, MLflow won't let you create a new experiment with the same name while the trashed one still exists; you have to purge it first with `mlflow gc --backend-store-uri sqlite:///mlflow.db`, which permanently deletes both the DB rows and the `mlartifacts/<id>/` directory.

2. **Configure a server-wide default-artifact-root that already uses names**, e.g. start the server with `mlflow ui --default-artifact-root <path>` and ensure new experiments get created with custom `artifact_location` from day one. This still doesn't help experiments that already exist with the old default.

### References

- [`mlflow.create_experiment` — `artifact_location` parameter](https://mlflow.org/docs/latest/python_api/mlflow.html#mlflow.create_experiment)
- [Artifact Stores — default-artifact-root and `mlflow-artifacts:` proxy](https://mlflow.org/docs/latest/ml/tracking/artifact-stores/)
- [MLflow 3 `LoggedModel` concept](https://mlflow.org/docs/latest/ml/model/)
